# Section 2 — Feature Engineering (WoE and IV)

# Notebook 02 — Feature Engineering: WoE and IV

## Central Question
**Which variables genuinely predict default — and how do we measure and quantify that precisely?**

## Why It Matters
Bivariate analysis in Notebook 01 showed us visually which variables relate
to default. But visual inspection is not enough for a regulatory model.

A regulator asks: how much information does each variable contribute?
Can you rank them objectively? Can you justify keeping or dropping each one
with a number rather than a judgment call?

Information Value answers these questions with a rigorous information-theoretic
measure. Weight of Evidence transforms the raw variables into a scale that
makes the logistic regression's linearity assumption valid by construction.

Get this step wrong and you feed the model variables that carry noise instead
of signal, or you force it to approximate non-linear relationships with a
linear function — producing a model that is both less accurate and less
interpretable than it should be.

## Key Finding
*To be completed after analysis.*

## Data
Loaded from `../data/X_train_clean.csv` and `../data/X_test_clean.csv`
produced by Notebook 01. All 11 features are already cleaned — no further
data quality intervention is required in this notebook.

In [19]:
from optbinning import BinningProcess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [20]:
# Load clean datasets produced by Notebook 01
X_train = pd.read_csv('../data/X_train_clean.csv')
X_test = pd.read_csv('../data/X_test_clean.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Train default rate: {y_train.mean():.4f}")
print(f"Test default rate:  {y_test.mean():.4f}")
print(f"\nFeatures:")
for col in X_train.columns:
    print(f"  {col}")

print(X_train['DebtRatio_valid'].head())

X_train shape: (120000, 11)
X_test shape:  (30000, 11)
Train default rate: 0.0668
Test default rate:  0.0668

Features:
  RevolvingUtilizationOfUnsecuredLines
  age
  NumberOfTime30-59DaysPastDueNotWorse
  MonthlyIncome
  NumberOfOpenCreditLinesAndLoans
  NumberOfTimes90DaysLate
  NumberRealEstateLoansOrLines
  NumberOfTime60-89DaysPastDueNotWorse
  NumberOfDependents
  Income_missing
  DebtRatio_valid
0    0.291084
1    0.498553
2    0.211999
3    0.291084
4    1.017328
Name: DebtRatio_valid, dtype: float64


In [21]:
# Separate continuous and binary features
# Binary features have only 2 unique values
binary_features = [col for col in X_train.columns 
                   if X_train[col].nunique() == 2]
continuous_features = [col for col in X_train.columns 
                       if X_train[col].nunique() > 2]

print(f"Continuous features ({len(continuous_features)}): {continuous_features}")
print(f"Binary features ({len(binary_features)}): {binary_features}")

Continuous features (10): ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'DebtRatio_valid']
Binary features (1): ['Income_missing']


In [22]:
def calculate_iv_manual(x, y, feature_name, smoothing=0.5):
    """
    Manual WoE and IV calculation for binary variables.
    Used when OptimalBinning fails due to too few unique values.
    """
    df = pd.DataFrame({'x': x, 'y': y})
    
    total_good = (df['y'] == 0).sum()
    total_bad = (df['y'] == 1).sum()
    
    grouped = df.groupby('x')['y'].agg(
        total='count', bad='sum'
    ).reset_index()
    grouped.columns = ['bin', 'total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    
    n_bins = len(grouped)
    grouped['dist_good'] = (grouped['good'] + smoothing) / (total_good + smoothing * n_bins)
    grouped['dist_bad'] = (grouped['bad'] + smoothing) / (total_bad + smoothing * n_bins)
    grouped['woe'] = np.log(grouped['dist_good'] / grouped['dist_bad'])
    grouped['iv_bin'] = (grouped['dist_good'] - grouped['dist_bad']) * grouped['woe']
    
    total_iv = grouped['iv_bin'].sum()
    woe_map = dict(zip(grouped['bin'], grouped['woe']))
    
    return total_iv, woe_map, grouped


# Calculate IV for all features
results = []
fitted_binners = {}  # Store for transformation step

y_vals = y_train.values.astype(int)

# Continuous features — OptimalBinning
for feature in continuous_features:
    try:
        x = X_train[feature].values
        
        optb = OptimalBinning(
            name=feature,
            dtype='numerical',
            max_n_bins=10,
            min_bin_size=0.05
        )
        optb.fit(x, y_vals)
        
        binning_table = optb.binning_table.build()
        iv = binning_table['IV'].iloc[-1]
        
        results.append({
            'feature': feature,
            'iv': round(float(iv), 4),
            'method': 'OptimalBinning'
        })
        fitted_binners[feature] = {'type': 'optbinning', 'binner': optb}
        
    except Exception as e:
        # Fallback to manual for any failures
        iv, woe_map, _ = calculate_iv_manual(
            X_train[feature].values, y_vals, feature
        )
        results.append({
            'feature': feature,
            'iv': round(iv, 4),
            'method': 'Manual (fallback)'
        })
        fitted_binners[feature] = {'type': 'manual', 'woe_map': woe_map}

# Binary features — manual calculation always
for feature in binary_features:
    iv, woe_map, woe_table = calculate_iv_manual(
        X_train[feature].values, y_vals, feature
    )
    results.append({
        'feature': feature,
        'iv': round(iv, 4),
        'method': 'Manual (binary)'
    })
    fitted_binners[feature] = {'type': 'manual', 'woe_map': woe_map}
    print(f"\n[{feature}] Manual WoE table:")
    print(woe_table[['bin', 'total', 'bad', 'woe', 'iv_bin']])

# Results
iv_df = pd.DataFrame(results).sort_values('iv', ascending=False).reset_index(drop=True)
print("\nIV RANKING:")
print(iv_df.to_string(index=False))


[Income_missing] Manual WoE table:
   bin  total   bad       woe    iv_bin
0    0  94474  6634 -0.052893  0.002254
1    1  25526  1387  0.220213  0.009383

IV RANKING:
                             feature     iv          method
RevolvingUtilizationOfUnsecuredLines 1.1123  OptimalBinning
             NumberOfTimes90DaysLate 0.7958  OptimalBinning
NumberOfTime30-59DaysPastDueNotWorse 0.6786  OptimalBinning
                                 age 0.2512  OptimalBinning
     NumberOfOpenCreditLinesAndLoans 0.0870  OptimalBinning
                     DebtRatio_valid 0.0814  OptimalBinning
                       MonthlyIncome 0.0784  OptimalBinning
        NumberRealEstateLoansOrLines 0.0581  OptimalBinning
                  NumberOfDependents 0.0342  OptimalBinning
                      Income_missing 0.0116 Manual (binary)
NumberOfTime60-89DaysPastDueNotWorse 0.0000  OptimalBinning


In [25]:
print("Columns in X_train:")
print(X_train.columns.tolist())

print("\nFeatures in fitted_binners:")
print(list(fitted_binners.keys()))

print("\nMissing from X_train:")
print([f for f in fitted_binners.keys() if f not in X_train.columns])

print("\nExtra in X_train:")
print([f for f in X_train.columns if f not in fitted_binners.keys()])

Columns in X_train:
['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'Income_missing', 'DebtRatio_valid']

Features in fitted_binners:
['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'DebtRatio_valid', 'Income_missing']

Missing from X_train:
[]

Extra in X_train:
[]


In [27]:
X_train_woe = X_train.copy()
X_test_woe = X_test.copy()

for feature, binner_info in fitted_binners.items():
    
    if binner_info['type'] == 'optbinning':
        optb = binner_info['binner']
        
        try:
            # Transform using values only — avoids internal name lookup
            train_vals = X_train[feature].values.astype(float)
            test_vals = X_test[feature].values.astype(float)
            
            X_train_woe[feature] = optb.transform(train_vals, metric='woe')
            X_test_woe[feature] = optb.transform(test_vals, metric='woe')
            
        except Exception as e:
            # Fallback — extract WoE mapping manually from binning table
            print(f"[{feature}] optb.transform failed: {e}")
            print(f"[{feature}] Falling back to manual mapping")
            
            try:
                bt = optb.binning_table.build()
                # Build manual WoE map from binning table
                print(f"[{feature}] Binning table:")
                print(bt[['Bin', 'WoE']].to_string())
            except Exception as e2:
                print(f"[{feature}] Also failed to build table: {e2}")
    
    elif binner_info['type'] == 'manual':
        woe_map = binner_info['woe_map']
        X_train_woe[feature] = X_train[feature].map(woe_map).fillna(0.0)
        X_test_woe[feature] = X_test[feature].map(woe_map).fillna(0.0)

print("Transformation complete")
print(f"X_train_woe shape: {X_train_woe.shape}")

[DebtRatio_valid] optb.transform failed: 'DebtRatio_valid'
[DebtRatio_valid] Falling back to manual mapping
[DebtRatio_valid] Binning table:
                 Bin       WoE
0       (-inf, 0.14)  0.086781
1       [0.14, 0.23)   0.12116
2       [0.23, 0.27)   0.34507
3       [0.27, 0.35)  0.216328
4       [0.35, 0.42)   0.06764
5       [0.42, 0.51) -0.092034
6       [0.51, 0.74) -0.353997
7        [0.74, inf) -0.676944
8            Special       0.0
9            Missing       0.0
Totals                        
Transformation complete
X_train_woe shape: (120000, 11)


In [28]:
# Load raw data again for diagnosis
data_df = pd.read_csv('../data/data.csv')

# Check index alignment
print(f"X_train index range: {X_train.index.min()} to {X_train.index.max()}")
print(f"data_df index range: {data_df.index.min()} to {data_df.index.max()}")

# Check Income_missing flag
print(f"\nIncome_missing value counts:")
print(X_train['Income_missing'].value_counts())

# Check DebtRatio_valid
print(f"\nDebtRatio_valid missing: {X_train['DebtRatio_valid'].isna().sum()}")
print(f"DebtRatio_valid zeros: {(X_train['DebtRatio_valid'] == 0).sum()}")
print(f"DebtRatio_valid describe:")
print(X_train['DebtRatio_valid'].describe())

X_train index range: 0 to 119999
data_df index range: 0 to 149999

Income_missing value counts:
Income_missing
0    94474
1    25526
Name: count, dtype: int64

DebtRatio_valid missing: 0
DebtRatio_valid zeros: 1890
DebtRatio_valid describe:
count    120000.000000
mean          0.342499
std           0.293255
min           0.000000
25%           0.184688
50%           0.291084
75%           0.411677
max           1.986923
Name: DebtRatio_valid, dtype: float64
